In [ ]:
# Install dependencies
!pip install --upgrade nltk PyPDF2
!pip install PyPDF2

# Import libraries
import os
import re
import nltk
from PyPDF2 import PdfReader
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# NLTK Downloads
nltk.download('punkt')       # Tokenizer
nltk.download('stopwords')   # Stopwords
nltk.download('wordnet')     # Lemmatization
nltk.download('punkt_tab')

text = "This is a test sentence to check tokenization."
tokens = word_tokenize(text)
print(tokens)

['This', 'is', 'a', 'test', 'sentence', 'to', 'check', 'tokenization', '.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Step 1: Extract text from PDF
def extract_text_from_pdf(file_path):
    """
    Extracts text from a PDF file using PyPDF2.
    """
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        if not text.strip():
            raise ValueError("The PDF file is empty or contains unsupported content.")
        return text
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

In [ ]:
# Step 2: Clean, deduplicate
def clean_text(text):
    """
    Cleans the raw extracted text.
    Removes excessive whitespace, non-ASCII characters, and special characters.
    Converts to lowercase, removes duplicates, and filters out verbs.
    """
    # Basic cleaning
    text = re.sub(r'\s+', ' ', text.strip())
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Deduplicate
    unique_tokens = list(set(tokens))

    return unique_tokens

In [ ]:
# Step 3: Process a single resume
def process_single_resume(resume_path, job_description):
    """
    Handles the scenario where the user wants to match one resume with one job description.
    """
    raw_text = extract_text_from_pdf(resume_path)
    if raw_text:
        cleaned_resume = clean_text(raw_text)
        cleaned_job_description = clean_text(job_description)
        return cleaned_resume, cleaned_job_description
    return None, None

In [ ]:
# Step 4: Process multiple resumes
def process_multiple_resumes(folder_path, job_description):
    """
    Processes all resumes while displaying only one sample resume's text and cleaned keywords.
    """
    processed_resumes = {}
    sample_parsed_text = None  # Store sample resume text for display
    sample_cleaned_keywords = None  # Store sample cleaned keywords for display

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            raw_text = extract_text_from_pdf(file_path)
            if raw_text:
                cleaned_resume = clean_text(raw_text)
                processed_resumes[filename] = cleaned_resume

                # Capture the first resume as a sample only if not already captured
                if sample_parsed_text is None:
                    sample_parsed_text = raw_text[:500]  # First 500 characters as a sample
                    sample_cleaned_keywords = cleaned_resume

    cleaned_job_description = clean_text(job_description)
    return processed_resumes, cleaned_job_description, sample_parsed_text, sample_cleaned_keywords

In [ ]:
# Matching function
def calculate_match_percentage(resume_keywords, job_description_keywords):
    """
    Calculates the match percentage based on job description keywords.
    """
    # Find common keywords
    matching_keywords = set(resume_keywords) & set(job_description_keywords)
    match_percentage = (len(matching_keywords) / len(job_description_keywords)) * 100
    return round(match_percentage, 2), matching_keywords

In [ ]:
# Main program update
if __name__ == "__main__":
    print("Choose an option:")
    print("1. Match one resume with one job description (for an individual).")
    print("2. Match multiple resumes with one job description (for a company).")
    choice = input("Enter your choice (1 or 2): ")

    # The job description we'll match against
    job_description = input("\nEnter the job description:\n")

    if choice == "1":
        # Individual matching
        resume_path = input("\nEnter the full path to the resume PDF:\n")
        cleaned_resume, cleaned_job_description = process_single_resume(resume_path, job_description)

        if cleaned_resume and cleaned_job_description:
            match_percentage, matched_keywords = calculate_match_percentage(cleaned_resume, cleaned_job_description)

            # Make the output look nice with a boxed header
            print("\n")
            print("╔" + "═"*50 + "╗")  # Top of the box
            print("║" + "CANDIDATE MATCH ANALYSIS".center(50) + "║")  # Center the title in the box
            print("╚" + "═"*50 + "╝")  # Bottom of the box

            # Show all keywords found in the resume
            print("\nResume Keywords:")
            print(cleaned_resume)

            print("\nJob Description Keywords:")
            print(cleaned_job_description)

            print("\nMatching Keywords:")
            print(matched_keywords)

            print(f"\nMatch Percentage: {match_percentage}%")
            print("\n" + "═"*100)

    elif choice == "2":
        folder_path = input("\nEnter the folder path containing the resumes:\n")
        processed_resumes, cleaned_job_description, sample_parsed_text, sample_cleaned_keywords = process_multiple_resumes(folder_path, job_description)

        if processed_resumes and cleaned_job_description:
            # Display sample processed data
            print("\nCleaned Sample Resume Keywords List (First Resume Only):")
            print(", ".join(sorted(sample_cleaned_keywords)))

            print("\nCleaned Job Description Keywords List:")
            print(", ".join(sorted(cleaned_job_description)))

            # Rank resumes based on match percentage
            ranking = {}
            for filename, resume_keywords in processed_resumes.items():
                match_percentage, matched_keywords = calculate_match_percentage(resume_keywords, cleaned_job_description)
                ranking[filename] = (match_percentage, matched_keywords)

            # Sort resumes by match percentage (highest first)
            ranked_resumes = sorted(ranking.items(), key=lambda x: x[1][0], reverse=True)

            # Display header
            print("\n")
            print("╔" + "═" * 80 + "╗")
            print("║" + "CANDIDATE MATCHING RESULTS".center(80) + "║")
            print("╚" + "═" * 80 + "╝")

            print(f"\nScreening Summary: {len(ranked_resumes)} Applications Processed\n")

            # Format string for displaying results
            print("{:<6} {:<25} {:<15} {}".format("No.", "Resumes", "Match %", "Matching Keywords"))

            # Display ranked results with matched keywords as lists
            for i, (filename, (percentage, matched_keywords)) in enumerate(ranked_resumes, 1):
                print("{:<6} {:<25} {:<15} {}".format(
                    i,  # Ranking number
                    filename,  # Resume file name
                    f"{percentage:.2f}%",  # Match percentage
                    ", ".join((sorted(matched_keywords)))  # Matched keywords as list
                ))
                print()  # Add spacing

            # Final border line
            print("═" * 80)

    else:
        print("Invalid choice. Please restart the program.")

Choose an option:
1. Match one resume with one job description (for an individual).
2. Match multiple resumes with one job description (for a company).
Enter your choice (1 or 2): 1

Enter the job description:
Data Scientist (Contractor) Bangalore, IN Responsibilities We are looking for a capable data scientist to join the Analytics team, reporting locally in India Bangalore. This person’s responsibilities include research, design and development of Machine Learning and Deep Learning algorithms to tackle a variety of Fraud oriented challenges. The data scientist will work closely with software engineers and program managers to deliver end-to-end products, including: data collection in big scale and analysis, exploring different algorithmic approaches, model development, assessment and validation – all the way through production. Qualifications At least 3 years of hands-on development of complex Machine Learning models using modern frameworks and tools, ideally Python based. Solid under

In [ ]:
# Main program update
if __name__ == "__main__":
    print("Choose an option:")
    print("1. Match one resume with one job description (for an individual).")
    print("2. Match multiple resumes with one job description (for a company).")
    choice = input("Enter your choice (1 or 2): ")

    # The job description we'll match against
    job_description = input("\nEnter the job description:\n")

    if choice == "1":
        # Individual matching
        resume_path = input("\nEnter the full path to the resume PDF:\n")
        cleaned_resume, cleaned_job_description = process_single_resume(resume_path, job_description)

        if cleaned_resume and cleaned_job_description:
            match_percentage, matched_keywords = calculate_match_percentage(cleaned_resume, cleaned_job_description)

            # Make the output look nice with a boxed header
            print("\n")
            print("╔" + "═"*50 + "╗")  # Top of the box
            print("║" + "CANDIDATE MATCH ANALYSIS".center(50) + "║")  # Center the title in the box
            print("╚" + "═"*50 + "╝")  # Bottom of the box

            # Show all keywords found in the resume
            print("\nResume Keywords:")
            print(cleaned_resume)

            print("\nJob Description Keywords:")
            print(cleaned_job_description)

            print("\nMatching Keywords:")
            print(matched_keywords)

            print(f"\nMatch Percentage: {match_percentage}%")
            print("\n" + "═"*100)

    elif choice == "2":
        folder_path = input("\nEnter the folder path containing the resumes:\n")
        processed_resumes, cleaned_job_description, sample_parsed_text, sample_cleaned_keywords = process_multiple_resumes(folder_path, job_description)

        if processed_resumes and cleaned_job_description:
            # Display sample processed data
            print("\nCleaned Sample Resume Keywords List (First Resume Only):")
            print(", ".join(sorted(sample_cleaned_keywords)))

            print("\nCleaned Job Description Keywords List:")
            print(", ".join(sorted(cleaned_job_description)))

            # Rank resumes based on match percentage
            ranking = {}
            for filename, resume_keywords in processed_resumes.items():
                match_percentage, matched_keywords = calculate_match_percentage(resume_keywords, cleaned_job_description)
                ranking[filename] = (match_percentage, matched_keywords)

            # Sort resumes by match percentage (highest first)
            ranked_resumes = sorted(ranking.items(), key=lambda x: x[1][0], reverse=True)

            # Display header
            print("\n")
            print("╔" + "═" * 80 + "╗")
            print("║" + "CANDIDATE MATCHING RESULTS".center(80) + "║")
            print("╚" + "═" * 80 + "╝")

            print(f"\nScreening Summary: {len(ranked_resumes)} Applications Processed\n")

            # Format string for displaying results
            print("{:<6} {:<25} {:<15} {}".format("No.", "Resumes", "Match %", "Matching Keywords"))

            # Display ranked results with matched keywords as lists
            for i, (filename, (percentage, matched_keywords)) in enumerate(ranked_resumes, 1):
                print("{:<6} {:<25} {:<15} {}".format(
                    i,  # Ranking number
                    filename,  # Resume file name
                    f"{percentage:.2f}%",  # Match percentage
                    ", ".join((sorted(matched_keywords)))  # Matched keywords as list
                ))
                print()  # Add spacing

            # Final border line
            print("═" * 80)

    else:
        print("Invalid choice. Please restart the program.")

Choose an option:
1. Match one resume with one job description (for an individual).
2. Match multiple resumes with one job description (for a company).
Enter your choice (1 or 2): 2

Enter the job description:
Data Scientist (Contractor) Bangalore, IN Responsibilities We are looking for a capable data scientist to join the Analytics team, reporting locally in India Bangalore. This person’s responsibilities include research, design and development of Machine Learning and Deep Learning algorithms to tackle a variety of Fraud oriented challenges. The data scientist will work closely with software engineers and program managers to deliver end-to-end products, including: data collection in big scale and analysis, exploring different algorithmic approaches, model development, assessment and validation – all the way through production. Qualifications At least 3 years of hands-on development of complex Machine Learning models using modern frameworks and tools, ideally Python based. Solid under

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(resume_keywords, job_description_keywords):
    """
    Calculates similarity score and extracts matched keywords using
    Count Vectorizer and Cosine Similarity.
    """
    # Convert lists to strings
    resume_text = ' '.join(resume_keywords)
    job_description_text = ' '.join(job_description_keywords)

    vectorizer = CountVectorizer(binary=True)
    documents = [resume_text, job_description_text]
    count_matrix = vectorizer.fit_transform(documents)
    similarity_score = cosine_similarity(count_matrix[0:1], count_matrix[1:2])[0][0] * 100

    # Get matched keywords
    matched_keywords = list(set(resume_keywords) & set(job_description_keywords))

    return round(similarity_score, 2), matched_keywords

In [ ]:
# Main program update
if __name__ == "__main__":
    print("Choose an option:")
    print("1. Match one resume with one job description (for an individual).")
    print("2. Match multiple resumes with one job description (for a company).")
    choice = input("Enter your choice (1 or 2): ")

    # The job description we'll match against
    job_description = input("\nEnter the job description:\n")

    if choice == "1":
        # Individual matching
        resume_path = input("\nEnter the full path to the resume PDF:\n")
        cleaned_resume, cleaned_job_description = process_single_resume(resume_path, job_description)

        if cleaned_resume and cleaned_job_description:
            match_percentage, matched_keywords = calculate_cosine_similarity(cleaned_resume, cleaned_job_description)

            # Make the output look nice with a boxed header
            print("\n")
            print("╔" + "═"*50 + "╗")  # Top of the box
            print("║" + "CANDIDATE MATCH ANALYSIS with COSINE SIMILARITY".center(50) + "║")  # Center the title in the box
            print("╚" + "═"*50 + "╝")  # Bottom of the box

            # Show all keywords found in the resume
            print("\nResume Keywords:")
            print(cleaned_resume)

            print("\nJob Description Keywords:")
            print(cleaned_job_description)

            print("\nMatching Keywords:")
            print(matched_keywords)

            print(f"\nMatch Percentage: {match_percentage}%")
            print("\n" + "═"*100)

    elif choice == "2":
        folder_path = input("\nEnter the folder path containing the resumes:\n")
        processed_resumes, cleaned_job_description, sample_parsed_text, sample_cleaned_keywords = process_multiple_resumes(folder_path, job_description)

        if processed_resumes and cleaned_job_description:
            # Display sample processed data
            print("\nCleaned Sample Resume Keywords List (First Resume Only):")
            print(", ".join(sorted(sample_cleaned_keywords)))

            print("\nCleaned Job Description Keywords List:")
            print(", ".join(sorted(cleaned_job_description)))

            # Rank resumes based on match percentage
            ranking = {}
            for filename, resume_keywords in processed_resumes.items():
                match_percentage, matched_keywords = calculate_cosine_similarity(resume_keywords, cleaned_job_description)
                ranking[filename] = (match_percentage, matched_keywords)

            # Sort resumes by match percentage (highest first)
            ranked_resumes = sorted(ranking.items(), key=lambda x: x[1][0], reverse=True)

            # Display header
            print("\n")
            print("╔" + "═" * 80 + "╗")
            print("║" + "CANDIDATE MATCHING RESULTS with Cosine Similarity".center(80) + "║")
            print("╚" + "═" * 80 + "╝")

            print(f"\nScreening Summary: {len(ranked_resumes)} Applications Processed\n")

            # Format string for displaying results
            print("{:<6} {:<25} {:<15} {}".format("No.", "Resumes", "Cosine Similarity %", "Matching Keywords"))

            # Display ranked results with matched keywords as lists
            for i, (filename, (match_percentage, matched_keywords)) in enumerate(ranked_resumes, 1):
                print("{:<6} {:<25} {:<15} {}".format(
                    i,  # Ranking number
                    filename,  # Resume file name
                    f"{match_percentage:.2f}%",  # Match percentage
                    ", ".join((sorted(matched_keywords)))  # Matched keywords as list
                ))
                print()  # Add spacing

            # Final border line
            print("═" * 80)

    else:
        print("Invalid choice. Please restart the program.")

Choose an option:
1. Match one resume with one job description (for an individual).
2. Match multiple resumes with one job description (for a company).
Enter your choice (1 or 2): 1

Enter the job description:
Data Scientist (Contractor) Bangalore, IN Responsibilities We are looking for a capable data scientist to join the Analytics team, reporting locally in India Bangalore. This person’s responsibilities include research, design and development of Machine Learning and Deep Learning algorithms to tackle a variety of Fraud oriented challenges. The data scientist will work closely with software engineers and program managers to deliver end-to-end products, including: data collection in big scale and analysis, exploring different algorithmic approaches, model development, assessment and validation – all the way through production. Qualifications At least 3 years of hands-on development of complex Machine Learning models using modern frameworks and tools, ideally Python based. Solid under